# LoRA na 3 epokach — domknięcie pomiaru kosztu

Domyka pomiar kosztu po awarii `run("lora")` w `lora_vs_full_3ep`: torchao odinstalowany, peft przypięty do 0.19.1.

In [ ]:
!pip uninstall -y -q torchao 2>/dev/null
!pip install -q -U "transformers>=4.44,<5.2" "datasets>=2.20" accelerate "peft==0.19.1" 2>/dev/null
import torch, transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
try:
    import torchao; print("UWAGA: torchao nadal obecny:", torchao.__version__)
except ImportError:
    print("torchao usunięty — dyspozytor peft zostanie pominięty")

In [ ]:
import glob, gc, shutil, time, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import (f1_score, hamming_loss, jaccard_score, accuracy_score,
                             precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
warnings.filterwarnings("ignore")
RANDOM_STATE=42
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
OUT="/kaggle/working"
MODEL_NAME="allegro/herbert-large-cased"
MAX_LEN, EPOCHS, PATIENCE = 128, 3, 2

In [ ]:
def find_csv(n):
    h=glob.glob(f"/kaggle/input/**/{n}",recursive=True)
    if not h: raise FileNotFoundError(f"{n} — dołącz dataset pl-emotion-processed")
    return h[0]
tw={s:pd.read_csv(find_csv(f"twitteremo_{s}.csv")) for s in ("train","val","test")}
for d in tw.values(): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw["val"][EMOTIONS].values,tw["test"][EMOTIONS].values
print("train",len(tw["train"]),"val",len(tw["val"]),"test",len(tw["test"]))

In [ ]:
def evaluate(yt,yp):
    return {"f1_macro":f1_score(yt,yp,average="macro",zero_division=0),
            "f1_micro":f1_score(yt,yp,average="micro",zero_division=0),
            "precision_macro":precision_score(yt,yp,average="macro",zero_division=0),
            "recall_macro":recall_score(yt,yp,average="macro",zero_division=0),
            "hamming_loss":hamming_loss(yt,yp),
            "jaccard_macro":jaccard_score(yt,yp,average="macro",zero_division=0),
            "subset_accuracy":accuracy_score(yt,yp)}
def find_optimal_thresholds(yt,yp):
    thr=np.full(yt.shape[1],0.5)
    for i in range(yt.shape[1]):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            f=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[i]=bt
    return thr
def f1_macro_ci(yt,yp,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); n=len(yt)
    base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
pos=tw["train"][EMOTIONS].values.sum(0); neg=len(tw["train"])-pos
POS_WEIGHT=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),pos_weight=self.pw.to(out.logits.device))
        return (loss,out) if return_outputs else loss

def to_ds(df):
    d=Dataset.from_dict({"text":df["tekst"].tolist(),"labels":df[EMOTIONS].values.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns=["text"])
ds_train,ds_val,ds_test=to_ds(tw["train"]),to_ds(tw["val"]),to_ds(tw["test"])
cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}

In [ ]:
# TEST DYMNY: zbuduj adaptery LoRA i od razu je porzuć. Jeśli peft jest niezgodny,
# padnie tu po ~40 s, a nie po godzinie treningu.
from peft import LoraConfig, TaskType, get_peft_model
_m=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
_m=get_peft_model(_m, LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
                                 lora_dropout=0.1, target_modules=["query","value"]))
_m.print_trainable_parameters()
del _m; gc.collect(); torch.cuda.empty_cache()
print("test dymny OK — peft tworzy adaptery poprawnie")

In [ ]:
def run(variant):
    """Train one variant, return metrics with wall-clock minutes and peak VRAM."""
    torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=len(EMOTIONS),problem_type="multi_label_classification")

    if variant=="lora":
        from peft import LoraConfig, TaskType, get_peft_model
        model=get_peft_model(model, LoraConfig(
            task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
            lora_dropout=0.1, target_modules=["query","value"]))
        model.enable_input_require_grads()
        model.print_trainable_parameters()
        batch, lr = 8, 1e-4
    else:
        batch, lr = 16, 2e-5
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in model.parameters())

    args=TrainingArguments(output_dir=f"{OUT}/ckpt_{variant}",eval_strategy="epoch",
        save_strategy="epoch",save_total_limit=1,load_best_model_at_end=True,
        metric_for_best_model="f1_macro",greater_is_better=True,
        per_device_train_batch_size=batch,per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,gradient_checkpointing=True,
        num_train_epochs=EPOCHS,learning_rate=lr,warmup_ratio=0.1,weight_decay=0.01,
        fp16=True,logging_steps=200,report_to="none",seed=RANDOM_STATE,data_seed=RANDOM_STATE)
    trainer=WeightedTrainer(model=model,args=args,train_dataset=ds_train,eval_dataset=ds_val,
        data_collator=DataCollatorWithPadding(tok),compute_metrics=cm,pos_weight=POS_WEIGHT,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])

    t0=time.time(); trainer.train(); minutes=(time.time()-t0)/60
    peak_gb=torch.cuda.max_memory_allocated()/2**30

    p_val=expit(trainer.predict(ds_val).predictions)
    p_test=expit(trainer.predict(ds_test).predictions)
    thr=find_optimal_thresholds(y_val,p_val); pred=(p_test>=thr).astype(int)
    m=evaluate(y_test,pred); base,lo,hi=f1_macro_ci(y_test,pred)
    m.update({"wariant":variant,"epoki":EPOCHS,"batch":batch,"lr":lr,
              "minutes":round(minutes,1),"peak_vram_gb":round(peak_gb,2),
              "trenowane_wagi":trainable,"wszystkie_wagi":total,
              "udzial_trenowanych_pct":round(100*trainable/total,3),
              "ci_low":round(lo,3),"ci_high":round(hi,3)})
    np.save(f"{OUT}/proba_test_{variant}_3ep.npy",p_test)
    np.save(f"{OUT}/proba_val_{variant}_3ep.npy",p_val)
    print(f"\n[{variant}] F1-Macro={m['f1_macro']:.4f} [{lo:.3f}; {hi:.3f}]  "
          f"{minutes:.1f} min  peak VRAM {peak_gb:.2f} GB  trenowane {trainable/1e6:.1f}M")

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"{OUT}/ckpt_{variant}",ignore_errors=True)
    return m

In [ ]:
FULL={"wariant":"full","f1_macro":0.5823,"ci_low":0.535,"ci_high":0.625,
      "minutes":80.8,"peak_vram_gb":7.99,"trenowane_wagi":355_100_000,
      "udzial_trenowanych_pct":100.0,"zrodlo":"kernel lora-vs-full-3ep"}
lora=run("lora"); lora["zrodlo"]="ten kernel"
df=pd.DataFrame([FULL,lora])
df.to_csv(f"{OUT}/result.csv",index=False)
print(df[["wariant","f1_macro","minutes","peak_vram_gb","udzial_trenowanych_pct"]].to_string(index=False))
print(f"\nLoRA wobec pelnego dostrajania przy 3 epokach:")
print(f"  F1-Macro {lora['f1_macro']-FULL['f1_macro']:+.4f}")
print(f"  czas     {lora['minutes']-FULL['minutes']:+.1f} min  ({lora['minutes']/FULL['minutes']*100:.0f}% czasu pelnego)")
print(f"  VRAM     {lora['peak_vram_gb']-FULL['peak_vram_gb']:+.2f} GB  ({lora['peak_vram_gb']/FULL['peak_vram_gb']*100:.0f}% szczytu pelnego)")
print("\nUWAGA: pelne dostrajanie mierzone w osobnej sesji Kaggle (ta sama karta T4).")